## 🧪 Test Dataset

In [1]:
from my_datasets.imdb_dataset import IMDBDataset
from my_tokenizers.text_tokenizer import TextTokenizer

/home/ali/Projects/AI/mamba-imdb/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
imdb = IMDBDataset()

imdb.df.info()

Dataset downloaded to: /home/ali/.cache/kagglehub/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/versions/1
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 781.4 KB


In [3]:
imdb.df.describe()

,sentiment
count,50000.000000
mean,0.500000
std,0.500005
min,0.000000
25%,0.000000
50%,0.500000
75%,1.000000
max,1.000000


In [4]:
print(imdb.df["review"].head())

0    One of the other reviewers has mentioned that ...
1    A wonderful little production. The filming tec...
2    I thought this was a wonderful way to spend ti...
3    Basically there's a family where a little boy ...
4    Petter Mattei's "Love in the Time of Money" is...
Name: review, dtype: str


In [5]:
print(imdb.df["sentiment"].value_counts())

sentiment
1    25000
0    25000
Name: count, dtype: int64


## 🍰 Split data `80%` `10%` `10%`

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    imdb.df, test_size=0.2, stratify=imdb.df["sentiment"], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["sentiment"], random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("Train:")
print(train_df["sentiment"].value_counts())

print("\nValidation:")
print(val_df["sentiment"].value_counts())

print("\nTest:")
print(test_df["sentiment"].value_counts())

Train: 40000
Validation: 5000
Test: 5000
Train:
sentiment
1    20000
0    20000
Name: count, dtype: int64

Validation:
sentiment
0    2500
1    2500
Name: count, dtype: int64

Test:
sentiment
0    2500
1    2500
Name: count, dtype: int64


## 🎩 Train Tokenizer

In [7]:
print(train_df["review"].sample())

texts = train_df["review"].tolist()


25176    SPOILERS I love the Simpsons, and I have seen ...
Name: review, dtype: str


In [8]:
tokenizer = TextTokenizer(
    algorithm="wordlevel",
    vocab_size=20_000,
    min_frequency=2,
)

In [9]:
tokenizer.train(texts)
print("Vocabulary size:", tokenizer.vocab_size)

Vocabulary size: 20000


In [10]:
text = imdb.df["review"].iloc[0]

tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(tokens)
print(ids)
print(decoded)
print(tokenizer.vocab_size)

['One', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'Oz', 'episode', 'you', "'", 'll', 'be', 'hooked', '.', 'They', 'are', 'right', ',', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', '.', 'The', 'first', 'thing', 'that', 'struck', 'me', 'about', 'Oz', 'was', 'its', 'brutality', 'and', 'unflinching', 'scenes', 'of', 'violence', ',', 'which', 'set', 'in', 'right', 'from', 'the', 'word', 'GO', '.', 'Trust', 'me', ',', 'this', 'is', 'not', 'a', 'show', 'for', 'the', 'faint', 'hearted', 'or', 'timid', '.', 'This', 'show', 'pulls', 'no', 'punches', 'with', 'regards', 'to', 'drugs', ',', 'sex', 'or', 'violence', '.', 'Its', 'is', 'hardcore', ',', 'in', 'the', 'classic', 'use', 'of', 'the', 'word', '.', 'It', 'is', 'called', 'OZ', 'as', 'that', 'is', 'the', 'nickname', 'given', 'to', 'the', 'Oswald', '<UNK>', 'Security', 'State', '<UNK>', '.', 'It', 'focuses', 'mainly', 'on', '<UNK>', 'City', ',', 'an', 'experimental', 'se

## 🚜👷🚧🏗️ Data loader

In [11]:
from my_datasets.imdb_torch_dataset import IMDBTorchDataset

train_dataset = IMDBTorchDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
)

val_dataset = IMDBTorchDataset(
    dataframe=val_df,
    tokenizer=tokenizer,
)

test_dataset = IMDBTorchDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
)

In [12]:
from functools import partial

from torch.utils.data import DataLoader

from my_datasets.imdb_torch_dataset import collate_fn

batch_size = 32

collate = partial(collate_fn, pad_token_id=tokenizer.pad_token_id)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate,
)

print(f"Train dataset:      {len(train_dataset)}")
print(f"Validation dataset: {len(val_dataset)}")
print(f"Test dataset:       {len(test_dataset)}")

print(f"\nTrain batches:      {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

Train dataset:      40000
Validation dataset: 5000
Test dataset:       5000

Train batches:      1250
Validation batches: 157
Test batches:       157


## 💎 Train Model

### 🎯 define model

In [29]:
import torch
from torch import nn

from my_classifier import MambaClassifier
from my_trainers.trainer import Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MambaClassifier(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=128,
    d_state=64,
    headdim=32,
    num_classes=2,
    pad_token_id=tokenizer.pad_token_id
).to(device)

print(model)
print('Device:', device)

MambaClassifier(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (mamba): Mamba3(
    (in_proj): Linear(in_features=128, out_features=680, bias=False)
    (B_norm): RMSNorm()
    (C_norm): RMSNorm()
    (norm): RMSNorm()
    (out_proj): Linear(in_features=256, out_features=128, bias=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)
Device: cuda


### 🚀 Optimizer and 📈 LOSS

In [30]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

### 👟 🏋️ The `Trainer`

In [31]:
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
)

In [32]:
for epoch in range(5):
    train_loss, train_accuracy = trainer.train_epoch()
    
    val_loss, val_accuracy = trainer.evaluate(val_loader)
    
    print(
        f"Epoch {epoch}: "
        f"Train Loss={train_loss:.4f}, "
        f"Train Acc={train_accuracy:.4f}, "
        f"Val Loss={val_loss:.4f}, "
        f"Val Acc={val_accuracy:.4f}"
    )

Training: 100%|██████████| 1250/1250 [02:42<00:00,  7.68it/s]


Epoch 0: Train Loss=0.4173, Train Acc=0.8059, Val Loss=0.3134, Val Acc=0.8680


Training: 100%|██████████| 1250/1250 [02:43<00:00,  7.66it/s]


Epoch 1: Train Loss=0.2457, Train Acc=0.9020, Val Loss=0.2679, Val Acc=0.8892


Training: 100%|██████████| 1250/1250 [02:49<00:00,  7.37it/s]


Epoch 2: Train Loss=0.1502, Train Acc=0.9456, Val Loss=0.2817, Val Acc=0.8928


Training: 100%|██████████| 1250/1250 [02:47<00:00,  7.45it/s]


Epoch 3: Train Loss=0.0727, Train Acc=0.9750, Val Loss=0.3766, Val Acc=0.8878


Training: 100%|██████████| 1250/1250 [02:43<00:00,  7.67it/s]
                                                             

Epoch 4: Train Loss=0.0320, Train Acc=0.9890, Val Loss=0.4839, Val Acc=0.8854


In [33]:
test_loss, test_accuracy = trainer.evaluate(test_loader)
    
print(
        f"Test Loss={test_loss:.4f}, "
        f"Test Acc={test_accuracy:.4f}"
    )

Test Loss=0.4878, Test Acc=0.8892


## 💾 Save Parameters!

it will save model and tokenizer

In [17]:
torch.save(model.state_dict(), "mamba_imdb.pt")
tokenizer.save("mamba_imdb_tokenizer.json")

## 🔄 Load Params

In [18]:
import torch

from my_tokenizers.text_tokenizer import TextTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load tokenizer
tokenizer = TextTokenizer.load("mamba_imdb_tokenizer.json")

In [19]:
from my_classifier import MambaClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load model
state_dict = torch.load("mamba_imdb.pt", map_location=device)

loaded_model = MambaClassifier(
    vocab_size=tokenizer.vocab_size
).to(device)

loaded_model.load_weights("mamba_imdb.pt")

MambaClassifier(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (mamba): Mamba3(
    (in_proj): Linear(in_features=128, out_features=680, bias=False)
    (B_norm): RMSNorm()
    (C_norm): RMSNorm()
    (norm): RMSNorm()
    (out_proj): Linear(in_features=256, out_features=128, bias=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [20]:
loaded_model.predict_sentiment("The Movie was bad", tokenizer)

'Negative'

In [21]:
loaded_model.predict_sentiment("The Movie was Great", tokenizer)

'Positive'

## 💀 Our Must hard case

In [22]:
loaded_model.predict_sentiment(
    "A bad movie is a disaster, and a good movie is incredible. This was the latter.",
    tokenizer,
)

'Positive'

In [23]:
loaded_model.predict_sentiment(
    "A good movie is incredible, and a bad movie is a disaster. This was the former.",
    tokenizer,
)

'Positive'

In [24]:
import pandas as pd
import torch

benchmark_data = [
    # Negation
    ("This movie is not good at all.", "Negative"),
    ("I didn't dislike this movie.", "Positive"),
    ("It's hard to say that this movie isn't enjoyable.", "Positive"),
    ("The movie is not terrible, but it is far from great.", "Negative"),
    ("I can't say I was disappointed by this film.", "Positive"),

    # Contrast
    ("The story was weak, but the acting was absolutely fantastic.", "Positive"),
    ("The movie looked beautiful, yet the story was painfully boring.", "Negative"),
    ("I expected a terrible movie, but it turned out to be wonderful.", "Positive"),
    ("The beginning was promising, but the ending ruined everything.", "Negative"),
    ("Despite the mediocre acting, the movie was surprisingly enjoyable.", "Positive"),

    # Reference / Coreference
    ("The first movie was terrible and the second was excellent. The latter was unforgettable.", "Positive"),
    ("The first movie was excellent and the second was terrible. The former was unforgettable.", "Positive"),
    ("I watched two films. One was awful, but the other was fantastic.", "Positive"),
    ("The movie had many problems, but this was not the worst of them.", "Negative"),
    ("The acting was disappointing, but that was hardly the film's biggest problem.", "Negative"),

    # Sarcasm
    ("What a masterpiece! I almost fell asleep three times.", "Negative"),
    ("Brilliant acting — if making me laugh for all the wrong reasons counts.", "Negative"),
    ("I absolutely loved wasting two hours of my life on this.", "Negative"),
    ("Yes, because every great movie should have a plot this ridiculous.", "Negative"),
    ("A truly unforgettable experience, mostly because I wish I could forget it.", "Negative"),

    # Mixed sentiment
    ("The acting was excellent, but the story was so boring that I couldn't finish it.", "Negative"),
    ("The story was predictable, but the performances were so good that I loved the movie.", "Positive"),
    ("Beautiful cinematography cannot save such a terrible script.", "Negative"),
    ("The script has flaws, but the characters are so compelling that the movie works.", "Positive"),
    ("Everything about this movie is mediocre except the brilliant final act.", "Positive"),

    # Long-range dependency
    (
        "The director promised a thrilling experience. The cinematography was impressive, "
        "the actors were talented, and the soundtrack was beautiful. Unfortunately, "
        "none of these things could save the painfully boring story.",
        "Negative"
    ),
    (
        "I expected another forgettable action movie. The first thirty minutes did little "
        "to change my mind. However, by the end, I realized that this was one of the most "
        "entertaining films I had seen all year.",
        "Positive"
    ),
    (
        "The movie starts slowly. Very slowly. The characters seem uninteresting, the "
        "dialogue feels awkward, and the plot appears predictable. Then everything "
        "changes, and the final hour is absolutely fantastic.",
        "Positive"
    ),
    (
        "There are many things I could criticize about this movie. The pacing, the dialogue, "
        "the acting, and even the editing are questionable. Nevertheless, I genuinely enjoyed "
        "watching it.",
        "Positive"
    ),
    (
        "At first I thought this was going to be terrible. An hour later, despite everything "
        "I had disliked about the movie, I found myself completely absorbed in it.",
        "Positive"
    ),

    # Hard cases
    ("I wouldn't go so far as to call this movie bad.", "Positive"),
    ("I wouldn't exactly call this movie good.", "Negative"),
    ("It's not that the movie was bad; it's that it was completely forgettable.", "Negative"),
    ("The movie is far from perfect, but calling it bad would be unfair.", "Positive"),
    (
        "There was nothing particularly wrong with the movie, but there was nothing "
        "particularly good about it either.",
        "Negative"
    ),
    ("For a movie this predictable, it is surprisingly entertaining.", "Positive"),
    ("For a movie this promising, it is surprisingly disappointing.", "Negative"),
    ("I went in expecting to hate it and ended up loving it.", "Positive"),
    ("I went in expecting to love it and ended up regretting the entire experience.", "Negative"),
    ("The movie isn't without its flaws, but its strengths far outweigh them.", "Positive"),
]

In [25]:
def run_benchmark(model, tokenizer, benchmark_data):
    model.eval()

    results = []

    for index, (text, expected) in enumerate(benchmark_data, start=1):

        prediction = model.predict_sentiment(
            text,
            tokenizer
        )

        correct = prediction == expected

        results.append({
            "ID": index,
            "Sentence": text,
            "Expected": expected,
            "Prediction": prediction,
            "Correct": correct
        })

    return pd.DataFrame(results)

In [ ]:
benchmark_results = run_benchmark(
    loaded_model,
    tokenizer,
    benchmark_data
)
benchmark_results

    ID                                           Sentence  Expected  \
0    1                     This movie is not good at all.  Negative   
1    2                       I didn't dislike this movie.  Positive   
2    3  It's hard to say that this movie isn't enjoyable.  Positive   
3    4  The movie is not terrible, but it is far from ...  Negative   
4    5       I can't say I was disappointed by this film.  Positive   
5    6  The story was weak, but the acting was absolut...  Positive   
6    7  The movie looked beautiful, yet the story was ...  Negative   
7    8  I expected a terrible movie, but it turned out...  Positive   
8    9  The beginning was promising, but the ending ru...  Negative   
9   10  Despite the mediocre acting, the movie was sur...  Positive   
10  11  The first movie was terrible and the second wa...  Positive   
11  12  The first movie was excellent and the second w...  Positive   
12  13  I watched two films. One was awful, but the ot...  Positive   
13  14

In [ ]:
accuracy = benchmark_results["Correct"].mean()

print(f"Benchmark Accuracy: {accuracy:.2%}")

# LAB!

In [27]:
def analyze_token_coverage(texts, tokenizer):
    unk_token_id = tokenizer.tokenizer.tokenizer.token_to_id("<UNK>")

    total_tokens = 0
    unk_tokens = 0

    for text in texts:
        token_ids = tokenizer.encode(text)

        total_tokens += len(token_ids)

        unk_tokens += sum(
            token_id == unk_token_id
            for token_id in token_ids
        )

    known_tokens = total_tokens - unk_tokens

    unk_ratio = (
        unk_tokens / total_tokens
        if total_tokens > 0
        else 0.0
    )

    return {
        "Total tokens": total_tokens,
        "Known tokens": known_tokens,
        "<UNK> tokens": unk_tokens,
        "UNK ratio": unk_ratio
    }

In [28]:
texts = [sentence for sentence, _ in benchmark_data]

coverage = analyze_token_coverage(
    texts,
    tokenizer
)

print(f"Total tokens : {coverage['Total tokens']}")
print(f"Known tokens : {coverage['Known tokens']}")
print(f"<UNK> tokens : {coverage['<UNK> tokens']}")
print(f"UNK ratio    : {coverage['UNK ratio']:.2%}")

Total tokens : 653
Known tokens : 650
<UNK> tokens : 3
UNK ratio    : 0.46%
